# Klygo: Tesla T4 before / after benchmark

This notebook preserves the exact code run on Colab for the recorded T4 measurements.
Select a **T4 GPU** runtime, then run the cells in order. It downloads public model
weights and generates a controlled H.264 video from the public Ultralytics bus image.
No private files or account credentials are needed.

- Baseline: `fb0f42f4075cb2bfab250446c1df059a67b5b2bb`.
- Initial optimization: `9ff1abd3f1b39f594144d9a81373e7109998bf86`.
- The last experiment applies the exact follow-up patch in a third worktree.
- Source and weight hashes, all individual timings, numerical checks, and limitations
  are recorded in `metrics/t4.json`; the follow-up source delta is `metrics/t4_followup.diff`.
- Model loading, full-clip warmup, output hashing, and numerical verification are
  excluded from measured prediction time. CUDA is synchronized at the boundaries.
- Timings depend on the allocated runtime. Recorded results are observations, not guarantees.
- The native API is compared at batch 1. Do not compare FPS across different batch sizes.

The original browser download cell was omitted; reports are retained in the notebook
outputs and the printed artifacts directory. The code below has execution counts and
outputs cleared so it can be rerun; the final Markdown cell records the actual measurements.


In [ ]:
# Verify the actual accelerator before benchmarking.
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    raise RuntimeError('A CUDA GPU runtime is required for this benchmark.')

In [ ]:
# Install the same dependencies for both pinned Klygo revisions.
import os, sys, subprocess, tempfile
from pathlib import Path
ROOT = Path(tempfile.mkdtemp(prefix='klygo-gpu-', dir='/content'))
NEW = ROOT / 'optimized'
OLD = ROOT / 'baseline'
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir()
BASE_SHA = 'fb0f42f4075cb2bfab250446c1df059a67b5b2bb'
NEW_SHA = '9ff1abd3f1b39f594144d9a81373e7109998bf86'
def command(args):
    result = subprocess.run([str(x) for x in args], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(result.stdout[-6000:])
    return result.stdout
print(command(['git', 'clone', '--quiet', '--branch', 'perf/managed-input-pipeline', '--single-branch', 'https://github.com/IchigoMazone/klygo.git', NEW]))
command(['git', '-C', NEW, 'checkout', '--quiet', '--detach', NEW_SHA])
command(['git', '-C', NEW, 'worktree', 'add', '--detach', OLD, BASE_SHA])
print('Installing dependencies...', flush=True)
print(command([sys.executable, '-m', 'pip', 'install', '-q', '-e', NEW, 'ultralytics==8.4.208']))
import cv2, numpy as np, torch, ultralytics
assert torch.cuda.is_available(), 'CUDA GPU required'
WEIGHTS = ARTIFACTS / 'yolov8n.pt'
from ultralytics import YOLO
YOLO(str(WEIGHTS))
VIDEO = ARTIFACTS / 'bus_motion_h264.mp4'
asset = Path(ultralytics.__file__).parent / 'assets' / 'bus.jpg'
image = cv2.resize(cv2.imread(str(asset)), (1280, 720))
assert image is not None
encoder = subprocess.Popen(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', '1280x720', '-r', '30', '-i', '-', '-an', '-c:v', 'libx264', '-preset', 'veryfast', '-g', '60', '-bf', '2', '-pix_fmt', 'yuv420p', str(VIDEO)], stdin=subprocess.PIPE)
for index in range(240):
    frame = np.roll(image, int(20 * np.sin(index / 15)), axis=1).copy()
    cv2.putText(frame, str(index), (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    encoder.stdin.write(frame.tobytes())
encoder.stdin.close()
assert encoder.wait() == 0
print('GPU:', torch.cuda.get_device_name(0))
print('Versions:', torch.__version__, ultralytics.__version__, cv2.__version__)
print('Video: 240 frames, 1280x720, H.264, moving public bus sample (controlled workload).')
print('Baseline:', BASE_SHA)
print('Optimized:', NEW_SHA)
print('Artifacts:', ARTIFACTS)

In [ ]:
# Controlled CUDA benchmark: isolated processes, fixed settings, identical input.
import json, textwrap, time
WORKER = r'''
import gc, hashlib, json, os, statistics, sys, time
from pathlib import Path
import cv2, numpy as np, torch
from ultralytics import YOLO
from klygo import files, models
cfg = json.loads(sys.argv[1])
cv2.setNumThreads(1)
torch.set_num_threads(2)
torch.manual_seed(0)
assert torch.cuda.is_available()
torch.cuda.set_device(0)
torch.backends.cudnn.benchmark = True
options = dict(device='cuda:0', imgsz=640, half=False, rect=True, conf=0.25, iou=0.7, verbose=False)
if cfg['kind'] == 'raw':
    detector = YOLO(cfg['weights'])
    def predict():
        return detector.predict(cfg['video'], batch=cfg['batch'], vid_stride=cfg['stride'], **options)
else:
    # Both revisions use the same public configuration API and local model_id.
    # This avoids the baseline's unrelated loading-only weights forwarding bug.
    config_path = Path(cfg['out']).with_suffix('.model.json')
    models.export('yolov8n.pt', config_path, overwrite=True, device='cuda:0', imgsz=640,
                  half=False, model_rect=True, threshold=0.25, iou=0.7)
    metadata = files.load(config_path, verbose=False)
    metadata['model_id'] = cfg['weights']
    metadata['model'].pop('weights', None)
    detector = models.load(metadata)
    def predict():
        if cfg['kind'] == 'native':
            return detector.predict(cfg['video'], native=True, verbose=False)
        return detector.predict(cfg['video'], batch=cfg['batch'], workers=cfg['workers'],
                                prefetch=cfg['prefetch'], vid_stride=cfg['stride'], verbose=False)
def signature(result):
    digest = hashlib.sha256()
    count = objects = 0
    for frame in result:
        if cfg['kind'] == 'raw':
            coords = frame.boxes.xyxy.detach().cpu().numpy().astype(np.float64)
            scores = frame.boxes.conf.detach().cpu().numpy().astype(np.float64)
            ids = frame.boxes.cls.detach().cpu().numpy().astype(np.int32)
        else:
            coords = np.asarray(frame.boxes, dtype=np.float64)
            scores = np.asarray(frame.scores, dtype=np.float64)
            ids = np.asarray(frame.class_ids, dtype=np.int32)
        digest.update(np.round(coords, 3).tobytes())
        digest.update(np.round(scores, 5).tobytes())
        digest.update(ids.tobytes())
        objects += len(ids)
        count += 1
    return count, objects, digest.hexdigest()
# Full-clip warmup initializes the predictor, exact input shapes, and CUDA kernels.
warm = predict()
torch.cuda.synchronize()
del warm
gc.collect()
torch.set_num_threads(2)
seconds, signatures, counts, objects, peaks = [], [], [], [], []
for repeat in range(3):
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    result = predict()
    torch.cuda.synchronize()
    seconds.append(time.perf_counter() - started)
    peaks.append(torch.cuda.max_memory_allocated() / 2**20)
    # Parity checking is outside the timed interval for all modes.
    count, object_count, digest = signature(result)
    counts.append(count)
    objects.append(object_count)
    signatures.append(digest)
    del result
data = dict(name=cfg['name'], revision=cfg['revision'], kind=cfg['kind'], batch=cfg['batch'],
            workers=cfg['workers'], prefetch=cfg['prefetch'], stride=cfg['stride'],
            fps=counts[0] / statistics.median(seconds), seconds=seconds, frames=counts,
            objects=objects, signatures=signatures, stable=len(set(signatures)) == 1,
            peak_vram_mib=max(peaks), gpu=torch.cuda.get_device_name(0),
            torch=torch.__version__, torch_threads=torch.get_num_threads())
Path(cfg['out']).write_text(json.dumps(data, indent=2))
print('RESULT', json.dumps(data), flush=True)
'''
worker_path = ARTIFACTS / 'gpu_worker.py'
worker_path.write_text(textwrap.dedent(WORKER))
CASES = []
def add(name, revision, kind, batch=1, workers=1, prefetch=1, stride=1):
    CASES.append(dict(name=name, revision=revision, kind=kind, batch=batch, workers=workers,
                      prefetch=prefetch, stride=stride, weights=str(WEIGHTS), video=str(VIDEO),
                      out=str(ARTIFACTS / (name + '.json'))))
add('raw_b1', 'raw', 'raw')
for rev in ('baseline', 'optimized'):
    add(rev + '_native_b1', rev, 'native')
for rev in ('baseline', 'optimized'):
    add(rev + '_managed_b1_w1', rev, 'managed')
for rev in ('baseline', 'optimized'):
    add(rev + '_managed_b1_w4', rev, 'managed', workers=4, prefetch=2)
add('raw_b4', 'raw', 'raw', batch=4)
for rev in ('baseline', 'optimized'):
    add(rev + '_managed_b4_w4', rev, 'managed', batch=4, workers=4, prefetch=2)
add('raw_b4_stride3', 'raw', 'raw', batch=4, stride=3)
for rev in ('baseline', 'optimized'):
    add(rev + '_managed_b4_w4_stride3', rev, 'managed', batch=4, workers=4, prefetch=2, stride=3)
REPORTS = []
print('GPU:', torch.cuda.get_device_name(0))
print('All modes: imgsz=640, FP32, conf=0.25, IoU=0.7, rect=True; one full-clip warmup + 3 measured runs.')
print('Native is compared only at batch=1. Input: controlled H.264 bus-motion clip, not the original user video.')
for number, cfg in enumerate(CASES, 1):
    checkout = OLD if cfg['revision'] == 'baseline' else NEW
    env = os.environ.copy()
    env['PYTHONPATH'] = str(checkout)
    env['YOLO_VERBOSE'] = 'False'
    print(f"[{number}/{len(CASES)}] Running {cfg['name']} ...", flush=True)
    process = subprocess.run([sys.executable, str(worker_path), json.dumps(cfg)], cwd=ROOT,
                             env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             timeout=600)
    (ARTIFACTS / (cfg['name'] + '.log')).write_text(process.stdout)
    if process.returncode:
        print(process.stdout[-8000:])
        raise RuntimeError('Benchmark failed: ' + cfg['name'])
    result = json.loads(Path(cfg['out']).read_text())
    REPORTS.append(result)
    (ARTIFACTS / 'gpu_report.json').write_text(json.dumps(REPORTS, indent=2))
    print(f"  {result['fps']:.2f} FPS | frames={result['frames'][0]} | objects={result['objects'][0]} | stable={result['stable']}", flush=True)
import pandas as pd
summary = pd.DataFrame([{k: row[k] for k in ('name', 'fps', 'batch', 'workers', 'stride', 'stable', 'peak_vram_mib')} for row in REPORTS])
display(summary.round(2))
lookup = {row['name']: row for row in REPORTS}
for suffix in ('native_b1', 'managed_b1_w1', 'managed_b1_w4', 'managed_b4_w4', 'managed_b4_w4_stride3'):
    old, new = lookup['baseline_' + suffix], lookup['optimized_' + suffix]
    same = old['signatures'][0] == new['signatures'][0]
    print(f"{suffix}: {old['fps']:.2f} -> {new['fps']:.2f} FPS ({new['fps']/old['fps']:.2f}x), rounded output parity={same}")
print('Full report:', ARTIFACTS / 'gpu_report.json')


In [ ]:
# Persist machine-readable measurements in notebook output and show the comparison.
from IPython.display import Markdown, JSON, display
reports = json.loads((ARTIFACTS / 'gpu_report.json').read_text())
assert len(reports) == len(CASES), f'Only {len(reports)}/{len(CASES)} cases finished; inspect the previous cell.'
lookup = {r['name']: r for r in reports}
print('Pinned source verification:')
for checkout in (OLD, NEW):
    print(checkout.name, command(['git', '-C', checkout, 'rev-parse', 'HEAD']).strip())
    env = os.environ.copy()
    env['PYTHONPATH'] = str(checkout)
    print(subprocess.check_output([sys.executable, '-c', 'import klygo; print(klygo.__file__)'], env=env, cwd=ROOT, text=True))
display(JSON({'gpu': torch.cuda.get_device_name(0), 'video': str(VIDEO), 'frames': 240,
              'imgsz': 640, 'half': False, 'warmup': 'one full clip', 'repetitions': 3,
              'baseline_sha': BASE_SHA, 'optimized_sha': NEW_SHA, 'measurements': reports}, expanded=False))
lines = ['## Tesla T4: Klygo before / after',
         'Controlled H.264 clip: 240 frames, 1280x720; YOLOv8n, imgsz=640, FP32. Median of 3 runs after warmup.',
         '', '| Configuration | Raw FPS | Before FPS | After FPS | Change | Output parity |',
         '|---|---:|---:|---:|---:|---|']
for title, suffix, raw_name in [
    ('Native, batch 1', 'native_b1', 'raw_b1'),
    ('Managed, batch 1, workers 1', 'managed_b1_w1', 'raw_b1'),
    ('Managed, batch 1, workers 4', 'managed_b1_w4', 'raw_b1'),
    ('Managed, batch 4, workers 4', 'managed_b4_w4', 'raw_b4'),
    ('Managed, batch 4, workers 4, stride 3', 'managed_b4_w4_stride3', 'raw_b4_stride3')]:
    old, new, raw = lookup['baseline_' + suffix], lookup['optimized_' + suffix], lookup[raw_name]
    same = old['signatures'][0] == new['signatures'][0]
    lines.append(f"| {title} | {raw['fps']:.2f} | {old['fps']:.2f} | {new['fps']:.2f} | {(new['fps']/old['fps']-1)*100:+.1f}% | {'MATCH' if same else 'DIFF'} |")
lines += ['', '**Scope:** this is a controlled moving-image video, not the original video behind the 90/87/30 FPS observation. Raw stride-3 and managed stride-3 can select different starting frames; only before/after parity is asserted in that row.',
          '', 'All existing public API signatures were preserved. No results are inferred from CPU measurements.']
summary_md = chr(10).join(lines)
(ARTIFACTS / 'gpu_summary.md').write_text(summary_md)
display(Markdown(summary_md))


In [ ]:
# Confirm batch-1 timing in reverse order and inspect numeric output differences.
import statistics
capture_worker = WORKER.replace('count = objects = 0', 'count = objects = 0\n    captured = []')
capture_worker = capture_worker.replace('objects += len(ids)', 'captured.append((coords, scores, ids))\n        objects += len(ids)')
capture_worker = capture_worker.replace('return count, objects, digest.hexdigest()', "np.savez(cfg['capture'], lengths=np.asarray([len(x[1]) for x in captured]), boxes=np.concatenate([x[0] for x in captured]), scores=np.concatenate([x[1] for x in captured]), class_ids=np.concatenate([x[2] for x in captured]))\n    return count, objects, digest.hexdigest()")
confirm_path = ARTIFACTS / 'gpu_capture_worker.py'
CONFIRM = []
confirm_names = ['optimized_native_b1', 'baseline_native_b1', 'optimized_managed_b1_w1',
                 'baseline_managed_b1_w1', 'raw_b1',
                 'baseline_managed_b4_w4_stride3', 'optimized_managed_b4_w4_stride3']
for base_name in confirm_names:
    cfg = dict(next(item for item in CASES if item['name'] == base_name))
    cfg['name'] = base_name + '_confirm'
    cfg['out'] = str(ARTIFACTS / (cfg['name'] + '.json'))
    cfg['capture'] = str(ARTIFACTS / (base_name + '.npz'))
    code = capture_worker.replace('range(3)', 'range(1)') if cfg['stride'] == 3 else capture_worker
    confirm_path.write_text(textwrap.dedent(code))
    env = os.environ.copy()
    env['PYTHONPATH'] = str(OLD if cfg['revision'] == 'baseline' else NEW)
    env['YOLO_VERBOSE'] = 'False'
    print('Confirming', base_name, flush=True)
    process = subprocess.run([sys.executable, str(confirm_path), json.dumps(cfg)], cwd=ROOT,
                             env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=600)
    (ARTIFACTS / (cfg['name'] + '.log')).write_text(process.stdout)
    if process.returncode:
        print(process.stdout[-6000:])
        raise RuntimeError('Confirmation failed: ' + base_name)
    data = json.loads(Path(cfg['out']).read_text())
    CONFIRM.append(data)
    print(f"  {data['fps']:.2f} FPS", flush=True)
    (ARTIFACTS / 'gpu_confirmation.json').write_text(json.dumps(CONFIRM, indent=2))
NUMERIC_PARITY = []
for suffix in ('native_b1', 'managed_b1_w1', 'managed_b4_w4_stride3'):
    left = np.load(ARTIFACTS / ('baseline_' + suffix + '.npz'))
    right = np.load(ARTIFACTS / ('optimized_' + suffix + '.npz'))
    same_lengths = np.array_equal(left['lengths'], right['lengths'])
    same_classes = np.array_equal(left['class_ids'], right['class_ids'])
    if same_lengths and same_classes:
        box_error = float(np.max(np.abs(left['boxes'] - right['boxes']), initial=0))
        score_error = float(np.max(np.abs(left['scores'] - right['scores']), initial=0))
        within_tolerance = box_error <= 0.01 and score_error <= 1e-5
    else:
        box_error = score_error = None
        within_tolerance = False
    record = dict(configuration=suffix, same_frame_counts=same_lengths, same_class_ids=same_classes,
                  max_box_error_px=box_error, max_score_error=score_error, within_tolerance=within_tolerance)
    NUMERIC_PARITY.append(record)
    print(record)
(ARTIFACTS / 'gpu_numeric_parity.json').write_text(json.dumps(NUMERIC_PARITY, indent=2))


In [ ]:
# Final reproducible report, including the confirmation runs and numeric checks.
import hashlib, zipfile
from IPython.display import Markdown, display
primary = {r['name']: r for r in REPORTS}
confirm = {r['name'].removesuffix('_confirm'): r for r in CONFIRM}
parity = {r['configuration']: r for r in NUMERIC_PARITY}
final_lines = ['## GPU benchmark results - Tesla T4',
               'YOLOv8n, imgsz=640, FP32; controlled 240-frame 1280x720 H.264 video. Each main result is the median of 3 measured runs after full-clip warmup.',
               '', '| Mode | Raw FPS | Before FPS | After FPS | Repeat before / after | Output check |',
               '|---|---:|---:|---:|---|---|']
for title, suffix, raw_name in [
    ('Native, batch 1', 'native_b1', 'raw_b1'),
    ('Managed, batch 1, workers 1', 'managed_b1_w1', 'raw_b1'),
    ('Managed, batch 1, workers 4', 'managed_b1_w4', 'raw_b1'),
    ('Managed, batch 4, workers 4', 'managed_b4_w4', 'raw_b4'),
    ('Managed, batch 4, workers 4, stride 3', 'managed_b4_w4_stride3', 'raw_b4_stride3')]:
    before, after = primary['baseline_' + suffix], primary['optimized_' + suffix]
    strict = before['signatures'][0] == after['signatures'][0]
    numeric = parity.get(suffix)
    check = 'Exact rounded match' if strict else ('Within tolerance' if numeric and numeric['within_tolerance'] else 'Needs investigation')
    repeat = '--'
    if 'baseline_' + suffix in confirm:
        repeat = f"{confirm['baseline_' + suffix]['fps']:.2f} / {confirm['optimized_' + suffix]['fps']:.2f}"
    final_lines.append(f"| {title} | {primary[raw_name]['fps']:.2f} | {before['fps']:.2f} | {after['fps']:.2f} | {repeat} | {check} |")
final_lines += ['', '**Interpretation:** batch 4 with four workers improved strongly on this workload. Batch-1 results vary between runs and do not establish a stable improvement. Do not compare different batch sizes as if they were identical configurations.',
                '', 'Numeric tolerance: identical per-frame object counts and class-ID order; maximum coordinate error <= 0.01 pixel and maximum score error <= 0.00001. Strict hash differences alone can arise from floating-point differences; numeric checks are recorded below.',
                '', 'The video was generated from the public Ultralytics bus sample with small motion. It is not the original video behind the earlier 90/87/30 FPS observation. Raw stride-3 and managed stride-3 may start at different frames; compare the two Klygo revisions within that row.',
                '', '### Numeric output checks']
for record in NUMERIC_PARITY:
    final_lines.append('- ' + str(record))
final_lines += ['', f"Raw batch-1 control: {primary['raw_b1']['fps']:.2f} FPS initially; {confirm['raw_b1']['fps']:.2f} FPS in the confirmation run.",
                '', 'Baseline commit: ' + BASE_SHA, 'Optimized commit: ' + NEW_SHA]
FINAL_MD = chr(10).join(final_lines)
(ARTIFACTS / 'gpu_final_report.md').write_text(FINAL_MD)
manifest = dict(gpu=torch.cuda.get_device_name(0), torch=torch.__version__, ultralytics=ultralytics.__version__,
                opencv=cv2.__version__, baseline=BASE_SHA, optimized=NEW_SHA, video_frames=240,
                video_sha256=hashlib.sha256(VIDEO.read_bytes()).hexdigest(),
                weights_sha256=hashlib.sha256(WEIGHTS.read_bytes()).hexdigest(),
                primary=REPORTS, confirmation=CONFIRM, numeric_parity=NUMERIC_PARITY)
(ARTIFACTS / 'gpu_complete_report.json').write_text(json.dumps(manifest, indent=2))
BUNDLE = ARTIFACTS / 'klygo_t4_benchmark_results.zip'
with zipfile.ZipFile(BUNDLE, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(ARTIFACTS.iterdir()):
        if path.suffix in ('.json', '.md', '.npz', '.py'):
            bundle.write(path, path.name)
display(Markdown(FINAL_MD))
print('Downloadable report:', BUNDLE)


In [ ]:
# Validate the batch-1 follow-up without changing either pinned checkout.
FOLLOW = ROOT / 'followup'
if not FOLLOW.exists():
    command(['git', '-C', NEW, 'worktree', 'add', '--detach', FOLLOW, NEW_SHA])
source_path = FOLLOW / 'klygo/models/detection/base.py'
original_block = r'''        for part in batches:
            yield part, [materialize(image) for image in part]
        return

    stopped = Event()
'''
replacement_block = r'''        try:
            for part in batches:
                yield part, [materialize(image) for image in part]
        finally:
            close = getattr(iterator, "close", None)
            if callable(close):
                close()
        return

    source_prepares_pixels = (
        bool(getattr(images, "_predicates", ()))
        or bool(getattr(images, "_preprocessing_ops", ()))
        or getattr(images, "_motion_threshold", None) is not None
        or getattr(images, "_deduplicate_threshold", None) is not None
    )
    if batch == 1 and workers == 1 and not source_prepares_pixels:
        # One ordered worker already serializes video reads. Avoid a second
        # executor and a completed intermediate Future on every default frame.
        executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="klygo-decode")
        pending_single: deque[tuple[Any, Future[Any]]] = deque()
        exhausted = object()

        def submit_one() -> bool:
            image = next(iterator, exhausted)
            if image is exhausted:
                return False
            pending_single.append((image, executor.submit(materialize, image)))
            return True

        try:
            for _ in range(prefetch + 1):
                if not submit_one():
                    break
            while pending_single:
                image, future = pending_single.popleft()
                yield [image], [future.result()]
                submit_one()
        finally:
            for _image, future in pending_single:
                future.cancel()
            executor.shutdown(wait=True, cancel_futures=True)
            close = getattr(iterator, "close", None)
            if callable(close):
                close()
        return

    stopped = Event()
'''
source = source_path.read_text()
if original_block in source:
    source_path.write_text(source.replace(original_block, replacement_block))
else:
    assert replacement_block in source, 'Unexpected source state'
patch = command(['git', '-C', FOLLOW, 'diff', '--', 'klygo/models/detection/base.py'])
(ARTIFACTS / 'followup_pipeline.diff').write_text(patch)
print('Follow-up source SHA256:', hashlib.sha256(source_path.read_bytes()).hexdigest(), flush=True)
FOLLOWUP_REPORTS = []
followup_path = ARTIFACTS / 'gpu_followup_worker.py'
followup_path.write_text(textwrap.dedent(capture_worker))
jobs = [('followup_b1_a', 'followup', 'optimized_managed_b1_w1'),
        ('previous_b1_a', 'previous', 'optimized_managed_b1_w1'),
        ('previous_b1_b', 'previous', 'optimized_managed_b1_w1'),
        ('followup_b1_b', 'followup', 'optimized_managed_b1_w1'),
        ('followup_b4_w4', 'followup', 'optimized_managed_b4_w4')]
for name, revision, template in jobs:
    cfg = dict(next(item for item in CASES if item['name'] == template))
    cfg.update(name=name, revision=revision, out=str(ARTIFACTS / (name + '.json')),
               capture=str(ARTIFACTS / (name + '.npz')))
    checkout = FOLLOW if revision == 'followup' else NEW
    env = os.environ.copy()
    env['PYTHONPATH'] = str(checkout)
    env['YOLO_VERBOSE'] = 'False'
    print('Running', name, flush=True)
    process = subprocess.run([sys.executable, str(followup_path), json.dumps(cfg)], cwd=ROOT,
                             env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=600)
    (ARTIFACTS / (name + '.log')).write_text(process.stdout)
    if process.returncode:
        print(process.stdout[-6000:])
        raise RuntimeError('Follow-up benchmark failed: ' + name)
    result = json.loads(Path(cfg['out']).read_text())
    result['source_sha256'] = hashlib.sha256((checkout / 'klygo/models/detection/base.py').read_bytes()).hexdigest()
    result['source_ref'] = NEW_SHA + (' plus followup_pipeline.diff' if revision == 'followup' else '')
    FOLLOWUP_REPORTS.append(result)
    (ARTIFACTS / 'gpu_followup.json').write_text(json.dumps(FOLLOWUP_REPORTS, indent=2))
    print(f"  {result['fps']:.2f} FPS", flush=True)
previous_times = [t for r in FOLLOWUP_REPORTS if r['revision'] == 'previous' for t in r['seconds']]
followup_times = [t for r in FOLLOWUP_REPORTS if r['revision'] == 'followup' and r['batch'] == 1 for t in r['seconds']]
print('Pooled batch-1 median:', 240 / statistics.median(previous_times), '->', 240 / statistics.median(followup_times))
left, right = np.load(ARTIFACTS / 'previous_b1_b.npz'), np.load(ARTIFACTS / 'followup_b1_b.npz')
FOLLOWUP_PARITY = dict(same_frame_counts=np.array_equal(left['lengths'], right['lengths']),
                      same_class_ids=np.array_equal(left['class_ids'], right['class_ids']),
                      max_box_error_px=float(np.max(np.abs(left['boxes']-right['boxes']), initial=0)),
                      max_score_error=float(np.max(np.abs(left['scores']-right['scores']), initial=0)))
print('Follow-up parity:', FOLLOWUP_PARITY)


In [ ]:
# Export actual measurements; no model weights, private data, or browser state.
benchmark_export = dict(manifest)
benchmark_export['protocol'] = dict(model='YOLOv8n', imgsz=640, half=False, rect=True, conf=0.25, iou=0.7,
                                  device='cuda:0', warmup_full_clips=1, measured_repetitions=3,
                                  opencv_threads=1, torch_threads=2, cudnn_benchmark=True,
                                  timing='CUDA-synchronized prediction only; parity checks outside timing',
                                  workload='240-frame 1280x720 H.264 moving public bus sample, GOP 60, B-frames 2',
                                  native_batch=1, raw_stride3_start_may_differ=True)
benchmark_export['followup'] = dict(base_commit=NEW_SHA,
    source_sha256=hashlib.sha256(source_path.read_bytes()).hexdigest(),
    patch_sha256=hashlib.sha256(patch.encode()).hexdigest(),
    measurements=FOLLOWUP_REPORTS,
    pooled_previous_batch1_fps=240/statistics.median(previous_times),
    pooled_followup_batch1_fps=240/statistics.median(followup_times),
    parity=FOLLOWUP_PARITY)
benchmark_export['limitations'] = ['Controlled generated clip, not the original user video.',
    'Batch-1 measurements vary between processes; do not claim a universal speedup.',
    'Raw stride-3 may begin at a different frame; only compare the Klygo revisions for parity.',
    'Initial strict hash differences were investigated with numerical comparisons.',
    'Follow-up is identified by base commit plus exact source and patch hashes.']
print(json.dumps(benchmark_export, indent=2))


## Recorded measurements

Tesla T4; PyTorch 2.14.1+cu130; Ultralytics 8.4.208; OpenCV 4.14.0.
YOLOv8n, imgsz=640, FP32; 240 frames, 1280x720; full-clip warmup and three timed repetitions.

| Configuration | Raw FPS | Baseline FPS | Initial optimization FPS |
| --- | ---: | ---: | ---: |
| Native, batch 1 | 85.86 | 84.69 | 78.09 |
| Managed, batch 1, workers 1 | 85.86 | 75.13 | 64.05 |
| Managed, batch 1, workers 4 | 85.86 | 76.14 | 64.57 |
| Managed, batch 4, workers 4 | 159.68 | 27.48 | 122.30 |
| Managed, batch 4, workers 4, stride 3 | 97.34 | 7.91 | 84.64 |

Batch-1 confirmation runs varied, including reversed comparisons; all timings are
retained in the JSON report rather than selecting only favorable runs. Strict hash
differences were followed by numerical comparisons: per-frame counts and class IDs
matched; the largest measured coordinate error was 0.00048828125 pixel and score
error was 0.0000013709068298339844.

### Final single-worker follow-up

Interleaved order: follow-up A, previous A, previous B, follow-up B; three timings per
process. Pooled over six timings per version:

- Previous optimization: 75.54568461090905 FPS.
- Follow-up: 77.34311744093024 FPS (about 2.4% higher in this run).
- Follow-up batch 4, workers 4: 128.66798054151445 FPS.
- Source SHA256: `5ac4d0bc083821a384d9517a6e2438d8f9ed4a7bee97c634905e29954589241e`.

The small batch-1 difference is not evidence of a universal gain. The stronger result
is removal of reader-induced backward seeks in multi-worker video batches. The
controlled clip is not the original video behind the earlier 90/87/30 FPS observation.
Raw stride-3 can begin at a different frame; compare the two Klygo revisions for parity.
